# Parallel Workflow: Essay Evaluation with an LLM

Three LLM judges score the essay in parallel, then one node combines their feedback.

```
        -> evaluate_language ->
START   -> evaluate_analysis ->  final_evaluation -> END
        -> evaluate_clarity  ->
```

In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b')

In [2]:
# Each judge returns feedback + a score, so we ask the LLM for structured output
class Evaluation(BaseModel):
    feedback: str = Field(description='Short feedback on the essay')
    score: int = Field(description='Score out of 10', ge=0, le=10)


structured_model = model.with_structured_output(Evaluation)

In [3]:
class EssayState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    scores: Annotated[list[int], operator.add]  # all 3 judges write here; lists get added together
    avg_score: float

In [4]:
def evaluate_language(state: EssayState):
    result = structured_model.invoke(f"Evaluate the language quality of this essay and give a score out of 10.\n{state['essay']}")
    return {'language_feedback': result.feedback, 'scores': [result.score]}


def evaluate_analysis(state: EssayState):
    result = structured_model.invoke(f"Evaluate the depth of analysis of this essay and give a score out of 10.\n{state['essay']}")
    return {'analysis_feedback': result.feedback, 'scores': [result.score]}


def evaluate_clarity(state: EssayState):
    result = structured_model.invoke(f"Evaluate the clarity of thought of this essay and give a score out of 10.\n{state['essay']}")
    return {'clarity_feedback': result.feedback, 'scores': [result.score]}


def final_evaluation(state: EssayState):
    prompt = (f"Write a 2-3 line overall summary from this feedback.\n"
              f"Language: {state['language_feedback']}\n"
              f"Analysis: {state['analysis_feedback']}\n"
              f"Clarity: {state['clarity_feedback']}")
    overall = model.invoke(prompt).content
    avg = sum(state['scores']) / len(state['scores'])
    return {'overall_feedback': overall, 'avg_score': round(avg, 2)}

In [5]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_clarity', evaluate_clarity)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [6]:
essay = """India in the Age of AI
Artificial Intelligence is changing how India works, learns and governs. In healthcare,
AI helps doctors read scans faster in hospitals with few specialists. In agriculture,
farmers get weather and crop advice on their phones. Government services like Aadhaar
and UPI already run at a scale where AI can cut fraud and delays. But there are risks:
jobs in IT services and call centres may shrink, and data privacy laws are still young.
India's large pool of engineers is its biggest strength. If it invests in skilling,
clear regulation and local-language AI, it can be a leader rather than a follower."""

result = workflow.invoke({'essay': essay})

print('Scores:', result['scores'])
print('Average:', result['avg_score'])
print('Overall:', result['overall_feedback'])

Scores: [5, 8, 8]
Average: 7.0
Overall: The essay is clear, concise, and readable, effectively outlining AI’s benefits and risks in India. Its language is accessible, though some sentences feel terse and could use smoother transitions and varied structure. However, the analysis remains surface‑level; a deeper dive into drivers, policies, and comparative context would strengthen the argument.
